In [1]:
from pathlib import Path
import json
import os

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

In [2]:
PROJECT_ROOT = Path('.').resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'liar_binary.csv'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'

for path in [MODELS_DIR, REPORTS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

Use the original LIAR train/valid/test split to avoid accidental data leakage between training and evaluation.

In [3]:
df = pd.read_csv(DATA_PATH)

train_df = df[df['split'] == 'train'].copy()
valid_df = df[df['split'] == 'valid'].copy()
test_df = df[df['split'] == 'test'].copy()

X_train = train_df['statement'].fillna('')
y_train = train_df['label']

X_valid = valid_df['statement'].fillna('')
y_valid = valid_df['label']

X_test = test_df['statement'].fillna('')
y_test = test_df['label']

label_names = ['not_credible', 'credible']

In [4]:
def evaluate_binary_model(name: str, y_true, y_pred) -> dict[str, float | str]:
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average='binary',
        pos_label=1,
        zero_division=0,
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average='macro',
        zero_division=0,
    )
    return {
        'model': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_credible': precision,
        'recall_credible': recall,
        'f1_credible': f1,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
    }

In [5]:
def save_confusion_matrix(y_true, y_pred, title: str, output_path: Path) -> None:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    display.plot(values_format='d')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.show()

In [6]:
metrics_rows = []
prediction_frames = []

In [7]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_test)

In [8]:
metrics_rows.append(evaluate_binary_model('Dummy baseline', y_test, dummy_pred))
print(classification_report(y_test, dummy_pred, target_names=label_names, zero_division=0))

save_confusion_matrix(
    y_test,
    dummy_pred,
    'Dummy baseline confusion matrix',
    FIGURES_DIR / 'confusion_matrix_dummy.png',
)

              precision    recall  f1-score   support

not_credible       0.55      1.00      0.71       553
    credible       0.00      0.00      0.00       449

    accuracy                           0.55      1002
   macro avg       0.28      0.50      0.36      1002
weighted avg       0.30      0.55      0.39      1002



/var/folders/y0/z0x4blhs1hs31p1__yb1v57m0000gn/T/ipykernel_84856/111386560.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
joblib.dump(dummy, MODELS_DIR / 'dummy_baseline.joblib')
print(f"Sanity check! Dummy model file was succesfully saved: {os.path.exists(MODELS_DIR / 'dummy_baseline.joblib')}")

Sanity check! Dummy model file was succesfully saved: True


The majority-class dummy model is a minimum reference point and helps verify whether later models learn any claim-level signal beyond class imbalance.

TF-IDF converts each statement into weighted word and phrase features. Logistic Regression learns which phrases are more associated with credible or not credible statements. For a given prediction, the most influential phrases are approximated by coefficient x TF-IDF value.

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=30_000,
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=1.0,
    )),
])

model_tfidf.fit(X_train, y_train)
tfidf_pred = model_tfidf.predict(X_test)

In [11]:
metrics_rows.append(evaluate_binary_model("TF-IDF + Logistic Regression", y_test, tfidf_pred))
print(classification_report(y_test, tfidf_pred, target_names=label_names, zero_division=0))

save_confusion_matrix(
    y_test,
    tfidf_pred,
    "TF-IDF Logistic Regression confusion matrix",
    FIGURES_DIR / "confusion_matrix_tfidf.png",
)

              precision    recall  f1-score   support

not_credible       0.65      0.60      0.63       553
    credible       0.55      0.61      0.58       449

    accuracy                           0.60      1002
   macro avg       0.60      0.60      0.60      1002
weighted avg       0.61      0.60      0.61      1002



/var/folders/y0/z0x4blhs1hs31p1__yb1v57m0000gn/T/ipykernel_84856/111386560.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
joblib.dump(model_tfidf, MODELS_DIR / "tfidf_logreg.joblib")

['/Users/marcinmalek/Documents/Codes/Projects/WORK/news-credibility/models/tfidf_logreg.joblib']

In [13]:
vectorizer = model_tfidf.named_steps["tfidf"]
clf = model_tfidf.named_steps["clf"]
feature_names = vectorizer.get_feature_names_out()
coefs = clf.coef_[0]

coef_df = pd.DataFrame({
    "ngram": feature_names,
    "coefficient": coefs,
})

credible_terms = (
    coef_df.sort_values("coefficient", ascending=False)
    .head(20)
    .assign(direction="credible")
)
not_credible_terms = (
    coef_df.sort_values("coefficient", ascending=True)
    .head(20)
    .assign(direction="not_credible")
)

top_ngrams = pd.concat([credible_terms, not_credible_terms], ignore_index=True)
top_ngrams.to_csv(REPORTS_DIR / "tfidf_top_ngrams.csv", index=False)
top_ngrams

,ngram,coefficient,direction
0,percent,2.519287,credible
1,day,1.990772,credible
2,georgia,1.913797,credible
3,countries,1.753492,credible
4,half,1.699468,credible
5,times,1.641379,credible
6,average,1.633933,credible
7,months,1.574807,credible
8,million,1.565574,credible
9,terms,1.541399,credible


In [14]:
def explain_tfidf_prediction(text: str, top_n: int = 10) -> pd.DataFrame:
    transformed = vectorizer.transform([text])
    row = transformed.tocoo()
    contributions = []
    for _, feature_idx, tfidf_value in zip(row.row, row.col, row.data):
        contributions.append({
            "ngram": feature_names[feature_idx],
            "tfidf_value": tfidf_value,
            "coefficient": coefs[feature_idx],
            "contribution": tfidf_value * coefs[feature_idx],
        })
    return (
        pd.DataFrame(contributions)
        .sort_values("contribution", key=lambda s: s.abs(), ascending=False)
        .head(top_n)
    )

example_indices = test_df.sample(3, random_state=42).index
explanation_rows = []

for idx in example_indices:
    text = test_df.loc[idx, "statement"]
    pred = int(model_tfidf.predict([text])[0])
    explanation = explain_tfidf_prediction(text, top_n=8)
    for _, row in explanation.iterrows():
        explanation_rows.append({
            "statement_id": test_df.loc[idx, "statement_id"],
            "statement": text,
            "true_label": int(test_df.loc[idx, "label"]),
            "predicted_label": pred,
            **row.to_dict(),
        })

pd.DataFrame(explanation_rows).to_csv(REPORTS_DIR / "tfidf_example_explanations.csv", index=False)

This model uses a pretrained sentence-transformer to convert each short claim into dense semantic embeddings. A lightweight Logistic Regression classifier is trained on top of those embeddings. This satisfies the transformer/embedding-based part of the MVP without fine-tuning a large transformer.

In [15]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

X_train_emb = embedding_model.encode(X_train.tolist(), show_progress_bar=True, normalize_embeddings=True)
X_valid_emb = embedding_model.encode(X_valid.tolist(), show_progress_bar=True, normalize_embeddings=True)
X_test_emb = embedding_model.encode(X_test.tolist(), show_progress_bar=True, normalize_embeddings=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/254 [00:00<?, ?it/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [16]:
embedding_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=1.0,
    )),
])

embedding_clf.fit(X_train_emb, y_train)
embedding_pred = embedding_clf.predict(X_test_emb)

In [17]:
metrics_rows.append(evaluate_binary_model("Sentence embeddings + Logistic Regression", y_test, embedding_pred))
print(classification_report(y_test, embedding_pred, target_names=label_names, zero_division=0))

save_confusion_matrix(
    y_test,
    embedding_pred,
    "Sentence embeddings confusion matrix",
    FIGURES_DIR / "confusion_matrix_embeddings.png",
)

              precision    recall  f1-score   support

not_credible       0.68      0.62      0.65       553
    credible       0.58      0.64      0.61       449

    accuracy                           0.63      1002
   macro avg       0.63      0.63      0.63      1002
weighted avg       0.64      0.63      0.63      1002



/var/folders/y0/z0x4blhs1hs31p1__yb1v57m0000gn/T/ipykernel_84856/111386560.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
joblib.dump(
    {
        "embedding_model_name": EMBEDDING_MODEL_NAME,
        "classifier": embedding_clf,
        "notes": "Reload SentenceTransformer(embedding_model_name), encode statements, then call classifier.predict(...).",
    },
    MODELS_DIR / "transformer_or_embeddings.joblib",
)

['/Users/marcinmalek/Documents/Codes/Projects/WORK/news-credibility/models/transformer_or_embeddings.joblib']